# Predicting the **goal difference** with a Gaussian — a teaching notebook

A deliberately simple, end-to-end Bayesian workflow. Instead of modelling each
team's goals with a Poisson (what the main project does), here we model **one
number per match — the goal difference** — with a **Gaussian**.

### What you'll learn
1. How to turn raw match results into a **signed target** where the *sign is the result*.
2. How to write a small **Bayesian model** as a generative story (likelihood + priors).
3. How to **fit** it with MCMC and check that the fit is trustworthy.
4. How to **read** the parameters (team strengths, home advantage).
5. **How to use the posterior to predict brand-new games** — and where the
   uncertainty in that forecast comes from.

**Why the goal difference?**
$$\text{goal\_diff} = \text{home\_goals} - \text{away\_goals}$$
The *sign* is the result:

| goal_diff | meaning |
|---|---|
| $> 0$ | **home** team won |
| $= 0$ | draw |
| $< 0$ | **away** team won (a negative margin) |

**Why a Gaussian?** Goal difference is signed, roughly symmetric, and bell-shaped
around a small positive number (home edge). A Normal likelihood is the simplest
honest choice — and it makes the whole model a clean Bayesian linear model.

In [ ]:
# --- make the wc2026 package importable under any kernel ---
# (adds the repo's src/ to the path if the package isn't installed)
import sys, pathlib
try:
    import wc2026  # already installed? nothing to do
except ModuleNotFoundError:
    here = pathlib.Path.cwd()
    for _p in [here, *here.parents]:
        if (_p / 'src' / 'wc2026').is_dir():
            sys.path.insert(0, str(_p / 'src')); break

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pymc as pm
import arviz as az

from wc2026.data.sources import build_real_matches

RNG = 2026
plt.rcParams["figure.figsize"] = (8, 4)

## 1. Read the data

Real international results (2022 → today), already filtered to the 48 World Cup
teams by the project's loader. Each row is one played match.

In [ ]:
matches = build_real_matches(start="2022-01-01", end="2026-07-11")
matches = matches[["date", "home_team", "away_team", "home_goals", "away_goals"]].copy()
print(f"{len(matches)} matches, {matches['home_team'].nunique()} home teams")
matches.head()

## 2. The target variable: goal difference

We compute the target and look at its distribution. Watch the **sign**: the mass
just right of zero is the home advantage.

In [ ]:
matches["goal_diff"] = matches["home_goals"] - matches["away_goals"]

print(matches["goal_diff"].describe().round(2))

ax = matches["goal_diff"].plot.hist(bins=range(-8, 9), edgecolor="white")
ax.axvline(0, color="k", lw=1, ls="--")
ax.set_xlabel("goal difference  (home - away)")
ax.set_title("Target: signed goal difference  (<0 = away won,  >0 = home won)")
plt.show()

**Read the shape.** Roughly **symmetric and bell-shaped**, centred a little
above zero (the home edge) — exactly what a **Gaussian** describes well. Not
perfect (the target is integer-valued with slightly heavy tails), but a very
reasonable approximation for a simple, interpretable model.

## 3. Encode teams as integers

The model refers to teams by an index, so we map each name to a column number and
build integer arrays for the home side, away side, and the target.

In [ ]:
teams = sorted(set(matches["home_team"]) | set(matches["away_team"]))
team_idx = {t: i for i, t in enumerate(teams)}
n_teams = len(teams)

home_i = matches["home_team"].map(team_idx).to_numpy()
away_i = matches["away_team"].map(team_idx).to_numpy()
y = matches["goal_diff"].to_numpy().astype(float)

print(f"{n_teams} teams, {len(y)} observations")

## 4. The Bayesian model (the heart of the notebook)

A small **generative story**: *how would nature produce a goal difference?* Give
every team one latent number, its **strength** $s_t$. When $h$ hosts $a$, the
*expected* margin is the strength gap plus a **home advantage** $\alpha$; the real
margin scatters around it with noise $\sigma$.

**Likelihood (one row of data):**
$$\text{goal\_diff}_g \sim \mathcal{N}(\mu_g,\ \sigma),
\qquad \mu_g = \alpha + s_{h(g)} - s_{a(g)}$$

**Priors (beliefs before seeing data):**
$$
s_t \sim \mathcal{N}(0,\ \tau)\ \text{(strengths, shared scale }\tau),\quad
\alpha \sim \mathcal{N}(0.3,\ 0.5),\quad
\sigma \sim \text{HalfNormal}(3),\quad
\tau \sim \text{HalfNormal}(1)
$$

**Why each piece:** $\mu = \alpha + s_h - s_a$ is *linear in strengths* — only the
strength **difference** matters, which is what a goal *difference* should depend
on. $s_t \sim \mathcal{N}(0,\tau)$ softly centres strengths at zero (fixing the
"add a constant to everyone" ambiguity) and lets the data learn, via $\tau$, how
far apart teams really are — **partial pooling** that pulls thin-record teams
toward average. $\sigma$ is how noisy a single match is.

In [ ]:
with pm.Model() as model:
    # --- priors ---
    tau      = pm.HalfNormal("tau", 1.0)                 # spread of team strengths
    strength = pm.Normal("strength", 0.0, tau, shape=n_teams)
    home_adv = pm.Normal("home_adv", 0.3, 0.5)           # alpha
    sigma    = pm.HalfNormal("sigma", 3.0)               # match noise

    # --- expected margin for every match (vectorised) ---
    mu = home_adv + strength[home_i] - strength[away_i]

    # --- Gaussian likelihood on the observed goal difference ---
    pm.Normal("obs", mu=mu, sigma=sigma, observed=y)

    # --- fit with NUTS (Hamiltonian Monte Carlo) ---
    idata = pm.sample(1000, tune=1000, chains=4, target_accept=0.9,
                      random_seed=RNG)

## 5. Did it converge?

Check the sampler before trusting anything. **R-hat** should be $\approx 1.00$ and
there should be no divergence warnings.

In [ ]:
az.summary(idata, var_names=["home_adv", "sigma", "tau"]).round(3)

## 6. Read the parameters

The parameters are interpretable — the payoff of a simple model. `home_adv` is the
average home edge in goals; `sigma` is match noise; `strength` is one number per
team (higher = larger expected winning margin vs. an average opponent).

In [ ]:
strength_post = idata.posterior["strength"].mean(("chain", "draw")).to_numpy()
ranking = (pd.DataFrame({"team": teams, "strength": strength_post})
           .sort_values("strength", ascending=False).reset_index(drop=True))

top = ranking.head(12).iloc[::-1]
plt.barh(top["team"], top["strength"], color="#2b8cbe")
plt.xlabel("posterior mean strength  (expected margin vs. average team)")
plt.title("Team strength from the Gaussian goal-difference model")
plt.tight_layout(); plt.show()
ranking.head(10)

## 7. Using the posterior to predict **new** games

This is the section you asked about. The fit didn't give us one "best" set of
numbers — it gave us a **posterior**: a cloud of *thousands of plausible parameter
sets* $(s, \alpha, \sigma)$ that are all consistent with the data. To forecast a
match we push the **whole cloud** through the model, not a single point estimate.

Two important ideas:

**(a) We can predict *any* pairing — even one never played.** Because we rate
*teams* (not head-to-head matchups), a fixture the model has never seen — say a
knockout tie that doesn't exist yet — is priced from the two teams' strengths.

**(b) A forecast has *two* sources of uncertainty.** For a new game $h$ vs $a$:

1. **Parameter uncertainty** — we're not sure of the strengths, so the *expected*
   margin itself is a distribution:
   $$\mu^{(k)} = \alpha^{(k)} + s_h^{(k)} - s_a^{(k)} \quad \text{for each posterior draw } k$$
2. **Match noise** — even with strengths fixed, one match scatters by $\sigma$.

The **posterior-predictive** margin combines both:
$$\widetilde{\text{diff}}^{(k)} = \mu^{(k)} + \sigma^{(k)}\,\varepsilon^{(k)},
\qquad \varepsilon^{(k)} \sim \mathcal{N}(0,1)$$

Its **sign** gives win / draw / loss. Let's implement exactly this.

In [ ]:
def posterior_draws():
    """Flatten the posterior to arrays of shape (n_draws,) / (n_teams, n_draws)."""
    post = idata.posterior
    st = post["strength"].stack(s=("chain", "draw")).to_numpy()   # (n_teams, S)
    ha = post["home_adv"].stack(s=("chain", "draw")).to_numpy()   # (S,)
    sg = post["sigma"].stack(s=("chain", "draw")).to_numpy()      # (S,)
    return st, ha, sg


def predict_new_game(home, away, neutral=True, seed=0):
    """Return (mu, diff) for a new fixture:
       mu   = per-draw EXPECTED margin  (parameter uncertainty only)
       diff = posterior-PREDICTIVE margin (parameter uncertainty + match noise).
    """
    st, ha, sg = posterior_draws()
    mu = (0.0 if neutral else ha) + st[team_idx[home]] - st[team_idx[away]]
    rng = np.random.default_rng(seed)
    diff = mu + sg * rng.standard_normal(mu.shape)
    return mu, diff


# Example: a brand-new fixture. Compare the two distributions.
mu, diff = predict_new_game("Mexico", "South Korea", neutral=True)

plt.hist(mu,   bins=40, density=True, alpha=0.8, color="#d94801",
         label="expected margin  mu  (parameter uncertainty)")
plt.hist(diff, bins=40, density=True, alpha=0.5, color="#9ecae1",
         label="predictive margin  (mu + match noise)")
plt.axvline(0, color="k", ls="--", lw=1)
plt.xlabel("goal difference  (home - away)")
plt.title("Two sources of uncertainty: Mexico vs South Korea")
plt.legend(); plt.tight_layout(); plt.show()

print(f"E[margin] = {diff.mean():+.2f} goals")

**Notice** the blue (predictive) distribution is **much wider** than the
orange one. The orange width is *"how unsure are we about the two teams' true
strengths?"*; the extra blue width is *"even if we knew them, football is noisy."*
A good forecast reports the **wide** one — it's honest about both.

Now turn a predictive margin into **probabilities**. Because the Gaussian is
continuous, we call a margin within $\pm 0.5$ a **draw**, above $+0.5$ a home win,
below $-0.5$ an away win.

In [ ]:
def outcome_probs(diff):
    return {
        "E[margin]":  round(float(diff.mean()), 2),
        "P(home win)": round(float((diff > 0.5).mean()), 2),
        "P(draw)":     round(float((np.abs(diff) <= 0.5).mean()), 2),
        "P(away win)": round(float((diff < -0.5).mean()), 2),
    }


# Predict a whole slate of NEW games in one tidy table.
fixtures = [
    ("Mexico", "South Korea"),
    ("Brazil", "Argentina"),
    ("Spain", "France"),
    ("Morocco", "Portugal"),
    ("United States", "England"),
]

rows = []
for home, away in fixtures:
    _, diff = predict_new_game(home, away, neutral=True)
    rows.append({"home": home, "away": away, **outcome_probs(diff)})

pd.DataFrame(rows)

**How to read the table.** Each row is a forecast for a fixture the model
may never have seen. `E[margin]` is the expected goal difference (sign = favourite);
the three probabilities are the share of predictive draws landing in each outcome
band, and they sum to ~1. Flip `neutral=False` to add the home-field edge for a
true home game.

## 8. Try it yourself (exercises)

1. **Home advantage.** Re-run `predict_new_game("Mexico", "South Korea",
   neutral=False)` and compare `E[margin]` to the neutral version. How many goals
   is home advantage worth here?
2. **A never-played tie.** Predict a knockout fixture between two teams that never
   met in the data. It still works — why? (Hint: we rate *teams*, not matchups.)
3. **Decompose the uncertainty.** For a chosen fixture, print `mu.std()` and
   `diff.std()`. Which is bigger, and what does each represent?
4. **Sanity check.** Predict a strong team vs a weak one and confirm `P(home win)`
   is high and the predictive distribution sits well right of zero.

## 9. Takeaways — and how this differs from the scoreline model

**What we built.** A one-line Bayesian idea — *margin = home edge + strength gap +
noise* — that targets the **signed goal difference**, uses a **Gaussian** (so it's
a clean linear model), and still yields full win/draw/loss probabilities by pushing
new fixtures through the **posterior-predictive** distribution.

| | This notebook (Gaussian on the margin) | Main model (Poisson on goals) |
|---|---|---|
| Target | goal **difference** (one signed number) | each team's **goals** (two counts) |
| Gives | margin, win/draw/loss | full **scoreline** grid → 1X2, over/under, BTTS |
| Pros | dead simple, interpretable, models the result directly | respects that goals are counts; richer outputs |
| Cons | ignores that goals are integer counts; no scoreline | more moving parts |

**When to prefer this one:** teaching, a quick baseline, or when you only need the
margin/result. **When to prefer Poisson:** scorelines, exact totals, or simulating
a whole tournament of specific results.